# Repository execution setup
Use Python 3.10/3.11. For QLoRA select a CUDA GPU before installing. This notebook calls repository scripts; training code lives in `src/llm_grading/`.
Do not upload private student data to a public notebook. Keep outputs private and use persistent storage if the runtime may disconnect.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

# Ưu tiên nhận diện Kaggle để lưu vào /kaggle/working
if Path("/kaggle").exists():
    REPO = Path("/kaggle/working/LLM-APA")
elif Path("/content").exists():
    REPO = Path("/content/LLM-APA")
else:
    REPO = Path.cwd()

if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/duykhang2103/LLM-APA.git", str(REPO)], check=True)

os.chdir(REPO)

# Notebook kernels may use another Python version; repository scripts use Python 3.11.
subprocess.run([sys.executable, "-m", "pip", "install", "uv", "pyyaml"], check=True)
notebook_env = REPO / ".notebook-venv"
setup_env = dict(os.environ, UV_PROJECT_ENVIRONMENT=str(notebook_env))
subprocess.run([sys.executable, "-m", "uv", "sync", "--python", "3.11", "--group", "dev", "--group", "ml"], cwd=REPO, env=setup_env, check=True)
PYTHON = str(notebook_env / ("Scripts/python.exe" if os.name == "nt" else "bin/python"))

def run(script, *args):
    subprocess.run([PYTHON, str(REPO / "scripts" / script), *map(str, args)], cwd=REPO, check=True)

print("Đã thiết lập REPO tại:", REPO)

## Optional persistent storage
Set `PERSISTENT` below to a mounted volume, Google Drive, or a Kaggle output directory. On Colab you may mount Drive with the next cell. This is optional; no cloud SDK is used by the training code. Kaggle `/kaggle/working` must be exported/saved before ending the session.


In [ ]:
# Đảm bảo kết quả lưu vào /kaggle/working để Kaggle giữ lại file ở mục Output
PERSISTENT = Path("/kaggle/working/outputs")
PERSISTENT.mkdir(parents=True, exist_ok=True)

# Zero-shot / RAG experiments
P0 has no demonstrations. Freeze its prompt/model/settings, then switch to P1 training-only RAG. Prefer deterministic decoding and use the same validation IDs/evaluator as fine-tuning. Local Qwen defaults require CUDA; the API adapter uses an environment-variable key and explicit course data permission.


In [ ]:
import yaml
import json
import os
import getpass
from pathlib import Path

# ==========================================
# 1. CAU HINH TASK 2
# ==========================================
TASK = "task2"
METHOD = "zero_shot"
VERSION = "P1-openrouter-v1"

# Duong dan sample_dataset
DATA_ROOT = Path("/kaggle/input/datasets/duynguyen67/llm-contest/sample_dataset/sample_dataset")
if not DATA_ROOT.exists():
    for alt in [
        Path("/kaggle/input/llm-contest/sample_dataset/sample_dataset"),
        REPO / "sample_dataset",
    ]:
        if alt.exists():
            DATA_ROOT = alt
            break

# Doc config mac dinh cho task2
config = yaml.safe_load((REPO / f"configs/{TASK}/{METHOD}.yaml").read_text())
config["experiment"].update(id=f"{TASK}-{VERSION}", notes="Task 2 Gemma-4-31B via OpenRouter")
config["prompt"]["version"] = "v004"

config["data"].update(
    input_path=str(DATA_ROOT),
    prepared_dir=str(PERSISTENT / TASK / "prepared-v1"),
)
config["saving"]["output_dir"] = str(PERSISTENT / TASK / VERSION)

config.setdefault("task2", {}).update(
    salvage_labels=True,
    max_labels=3,
    fallback_on_failure=True,
    use_grading_policy=False,
)

# ==========================================
# 2. MODEL - Google Gemma 4 31B (FREE) qua OpenRouter
# ==========================================
if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass(
        "Nhap OpenRouter API Key (sk-or-v1-...): "
    )

config["model"] = {
    "provider": "api",
    "name_or_path": "google/gemma-4-31b-it:free",
    "base_url": "https://openrouter.ai/api/v1",
    "api_key_env": "OPENROUTER_API_KEY",
    "allow_private_data": True,
    "timeout_seconds": 90,
}

config["generation"].update(
    max_new_tokens=256,
    max_input_tokens=8192,
    do_sample=False,
    temperature=0,
)

CONFIG = PERSISTENT / f"{TASK}-{VERSION}-config.json"
CONFIG.write_text(json.dumps(config, indent=2))

# Chuan bi du lieu
run("prepare_data.py", "--config", CONFIG, "--training")
PREPARED = (
    Path(config["data"]["prepared_dir"])
    / ("config-" + config["experiment"]["id"] + ".json")
)
print("San sang chay Task 2:", PREPARED)


## Small smoke subset, then full validation
Smoke artifacts live under `smoke/`. RAG logs candidate exclusions/retrieved IDs; Task 1/2 examples exclude feedback. Task 3 filters to the requested level before ranking.


In [ ]:
# 1. Giữ lại smoke test để kiểm tra mô hình nạp vào GPU ổn định (chỉ mất vài giây)
run("smoke.py", "--config", PREPARED)
# run("predict.py", "--config", PREPARED, "--split", "val")
# OUT = Path(config["saving"]["output_dir"])
# run("validate_predictions.py", "--task", TASK, "--input", OUT / "predictions.json")
# run("evaluate.py", "--config", PREPARED, "--split", "val", "--predictions", OUT / "predictions.json")

## Inspect metrics and responses
Keep artifacts private. Raw model responses may quote student code. Macro-F1 uses all ten labels with zero_division=0; compare per-label support. Task 3 overlap/compliance are diagnostics requiring content review.


In [ ]:
# metrics = json.loads((OUT / "metrics.json").read_text())
# predictions = json.loads((OUT / "predictions.json").read_text())
# print(json.dumps(metrics, ensure_ascii=False, indent=2))
# predictions[:3]


## Final acceptance
Freeze settings before lecturer test data arrives. Configure its path and run prediction → validation → packaging without target inspection or retraining.


In [ ]:
import shutil
from pathlib import Path

OUT = Path(config["saving"]["output_dir"])
OUT.mkdir(parents=True, exist_ok=True)

# 1. Đường dẫn file đề thi Task 2 trong public_test
PUBLIC_TEST_FILE = Path("/kaggle/input/datasets/duynguyen67/llm-contest/public_test/public_test_release/task2_error_taxonomy.json")
if not PUBLIC_TEST_FILE.exists():
    for alt in [
        Path("/kaggle/input/llm-contest/public_test/public_test_release/task2_error_taxonomy.json"),
        Path("/kaggle/input/public_test/public_test_release/task2_error_taxonomy.json")
    ]:
        if alt.exists():
            PUBLIC_TEST_FILE = alt
            break

print("Đang chấm Task 2 trên đề thi:", PUBLIC_TEST_FILE)

# 2. Dự đoán phân loại lỗi cho toàn bộ 328 bài nộp
run("predict.py", 
    "--config", PREPARED, 
    "--input", str(PUBLIC_TEST_FILE), 
    "--output", str(OUT / "task2_public_predictions.json"))

# 3. Đóng gói theo format BTC yêu cầu (chuyển error_labels -> taxonomy_error)
run("package_predictions.py", 
    "--task", "task2", 
    "--input", str(OUT / "task2_public_predictions.json"), 
    "--output", str(OUT / "submission_task2.zip"))

# 4. Copy file zip ra ngoài root /kaggle/working để Kaggle hiện ngay trên giao diện Output
shutil.copy(OUT / "submission_task2.zip", "/kaggle/working/submission_task2.zip")
shutil.copy(OUT / "task2_public_predictions.json", "/kaggle/working/task2_public_predictions.json")

print("\n" + "="*50)
print("THÀNH CÔNG! File nộp bài Task 2 đã lưu tại:")
print("- /kaggle/working/submission_task2.zip")
print("="*50)